In [8]:
import numpy as np
from scipy import constants
import pandas as pd
pd.set_option('display.width', 10000) # Adjust for desired width
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None) # display full content in a column
import matplotlib.pyplot as plt
from tweezer_functions import * 
from IonChainTools import *
from scipy.optimize import fsolve
import matplotlib.colors as mcolors
import matplotlib.colorbar as mcolorbar
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import matplotlib.ticker as ticker
import itertools
import math

#Constants in SI units
eps0 = constants.epsilon_0 
m = 39.9626*constants.atomic_mass
c = constants.c
e = constants.e
hbar = constants.hbar
pi = np.pi

# setting up parameters that we're not changing
qubit_wavelength = 729e-9
tweezer_wavelength = 532e-9
omega_tweezer = 2*pi*c/tweezer_wavelength
print(omega_tweezer)
df = pd.read_csv("S_P_only.csv",sep = ",",encoding = "UTF-8")
lambdares = np.array(df["wavelength (nm)"])*1e-9
omega_res = 2*pi*c/lambdares
linewidths = np.array(df["A_ki (s^-1)"])
lifetimes = linewidths
print(linewidths)
#test

3540698434791077.0
[1.47e+08 1.40e+08]


In [37]:
e

1.602176634e-19

In [34]:
def ion_force_quartic(positions, N, k4, omega_rf_axial):
    """
    Calculate force on each ion.
    
    Inputs:
    -----------
    positions : array-like
        Position of each ion (length N)
    N : int
        Number of ions
    k4 : float
        Quartic interaction coefficient
    omega_rf_axial : float
        Axial trapping frequency from rf potential (2*pi*Hz)
    
    Returns:
    --------
    force_list : list
        Force on each ion (length N)
    """
    x = np.array(positions)
    N = len(x)
    A = 0.5 * m * omega_rf_axial**2
    B = (e**2) / (4 * pi * eps0)
    C = k4

    return [2*A*(x[m]) + 4*C * (x[m]**3) 
            - sum([B * (x[m] - x[n]) / (abs(x[m] - x[n])**3) for n in range(m) if x[m] != x[n]])  # Avoid division by zero
            + sum([B * (x[m] - x[n]) / (abs(x[m] - x[n])**3) for n in range(m+1, N) if x[m] != x[n]])  # Avoid division by zero
           
            for m in range(N)]


N = 15
k4 = 0.00177
omega_rf_axial = 150e3*2*pi
estimated_extreme = 0.481*N**0.765
x0 = np.linspace(-estimated_extreme, estimated_extreme, N)

ueq_test = fsolve(ion_force_quartic, x0, args=(N, k4, omega_rf_axial))
diff_list = []
for x, y in zip(ueq_test[0::], ueq_test[1::]):
    diff_list.append(y-x)
diff_list

[5.231334134260145e-06,
 4.323839194694244e-06,
 4.72025407683626e-06,
 3.44286201413891e-06,
 7.1912297104941415e-06,
 4.469411295911118e-06,
 -7.2870498116343415e-06,
 1.781096366938332e-05,
 3.011792890101129e-06,
 -7.898670131382756e-06,
 9.874248939254035e-06,
 2.2413323608826758e-06,
 3.211509994864699e-06,
 1.7354252790722454e-06]

In [ ]:
#using the duke paper to find what axial frequency I should use and what x4 should be:
#X2 = 0.00188 ,𝑋4 =0.00177
#1/2 m 𝜔𝑟^2  = E0/(d0^2) X2 
omega_axial = 0.0004403997921


[4.590577959967449e-05, 2.7355678378134504e-05]

the radial modes will be unchanged with the quartic potential, I just need to put in a d that makes equal spacing manually..and I'm not doing anything with axial modes right now so I don't need to bother really with anything else.  

In [ ]:
def mode_calc_a_quartic(m,omega_a_combined,ueq,N):
    """
    Hessian for ions in a pseudo-potential
    Inputs:
    ueq -- equilibrium positions of the ions (m)
    m -- mass of ion (kg)
    omega_a_combined -- combined radial frequency taking into account the rf potential as well as the tweezer potentials. 
                will look like array where each entry for untweezed ion is the rf radial frequency and each entry for the
                tweezed ions is sqrt(omega_tweezer^2 + omega_a_rf^2) (2*pi*Hz)
    omega_a -- axial trapping frequency created by rf potential (2*pi*Hz)
    
    Outputs: 
    modes -- list of tuples where each tuple is a mode (frequency [Hz], eigenvector)
    """

    A = np.zeros((N, N))
    coloumb = ((e**2) / (4 * pi * eps0))
    masses = np.array([m for _ in range(N)])
    for i in range(N):
        A[i][i] = (masses[i] * omega_a_combined[i]**2 + coloumb * sum(2 / (ueq[i] - ueq[m])**3 for m in range(0, i))
           + coloumb * sum(2 / (ueq[m] - ueq[i])**3 for m in range(i + 1, N)))# * masses[i]
        for j in range(0, i):
            A[i][j] = (-2/(ueq[i]-ueq[j])**3) *(coloumb)#* np.sqrt(masses[i])*np.sqrt(masses[j])
        for j in range(i+1, N):
            A[i][j] = (-2/ (ueq[j]-ueq[i])**3)*(coloumb)# *np.sqrt(masses[i])*np.sqrt(masses[j])

    eigvals, eigvecs = np.linalg.eig(A) # this gives eigenvalues and eigenvectors
    freqs =( np.sqrt(1*eigvals/m))/(2*pi) #eigenvalue = spring constant k, so freq = sqrt(e-val)/(2*pi*m)

    scaledmodes = [(f, v) for f, v in zip(freqs, eigvecs.T)]
    scaledmodes = sorted(scaledmodes, key=lambda mode: mode[0],reverse=False)
    modes = []
    for f, scaledvec in scaledmodes:
        vec = np.array([scaledvec[i]/1 for i in range(len(eigvals))])
        vec = vec / np.sqrt(vec.dot(vec))
        modes.append((f, vec))
     modes

Make a function that generates equally spaced ions for whatever N I want, and then use that function in all of my other functions.  see what else in the pipeline needs to change to accomodate this change.

In [ ]:
mode_calc_r_quartic(m,omega_r_combined,omega_rf_axial,k4,N,ueq_test)